# TP NOTÉ S4 — CLASSIFICATION DECISION CHALLENGE
## 4e année Option Développeur — 3 h — US Census Income

**Rendu :** un notebook Jupyter entièrement exécuté. Toute décision doit être justifiée par vos propres résultats.

**Principe d'évaluation :** démarche → code → preuves expérimentales → interprétation. Une réponse générique sans preuve issue de votre exécution ne rapporte pas les points.

## Mission — 5 pts

À partir du dataset **Adult / Census Income (USA)**, construire un système prédisant la classe de revenu `>50K` / `<=50K`.

Votre travail ne sera pas évalué uniquement sur l'accuracy. Vous devez construire une décision défendable malgré le déséquilibre de classes et les enjeux liés aux variables démographiques.

In [1]:
# Votre travail expérimental ici
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml

census = fetch_openml(name="adult", version=2, as_frame=True)
X_full = census.data
y_full = census.target

print(X_full.shape)
print(y_full.shape)

print(X_full.dtypes)
display(X_full.head())

distribution_cible = y_full.value_counts()
distribution_cible_pct = y_full.value_counts(normalize=True) * 100
print(distribution_cible)
print(distribution_cible_pct.round(2))

ratio_desequilibre = distribution_cible.max() / distribution_cible.min()
print(ratio_desequilibre)

print(X_full.isna().sum()[X_full.isna().sum() > 0])

colonnes_object = X_full.select_dtypes(include="object").columns
for col in colonnes_object:
    valeurs_suspectes = X_full[col].astype(str).str.strip().eq("?").sum()
    if valeurs_suspectes > 0:
        print(col, valeurs_suspectes, valeurs_suspectes / len(X_full) * 100)

(48842, 14)
(48842,)
age                  int64
workclass         category
fnlwgt               int64
education         category
education-num        int64
marital-status    category
occupation        category
relationship      category
race              category
sex               category
capital-gain         int64
capital-loss         int64
hours-per-week       int64
native-country    category
dtype: object


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States


class
<=50K    37155
>50K     11687
Name: count, dtype: int64
class
<=50K    76.07
>50K     23.93
Name: proportion, dtype: float64
3.179173440574998
workclass         2799
occupation        2809
native-country     857
dtype: int64


In [2]:
## Mission

# La base contient 48842 observations et 14 variables explicatives.

# La variable cible est `class`, qui correspond à la classe de revenu de l'individu
# (`<=50K` ou `>50K`).

# La cible est déséquilibrée : 76,07% des individus sont `<=50K` contre 23,93%
# pour `>50K`, soit un ratio d'environ 3,18.

# Un modèle qui prédirait systématiquement `<=50K` atteindrait déjà environ 76%
# d'accuracy, ce qui rend cette métrique insuffisante pour juger les modèles.

# Les variables explicatives sont numériques ou catégorielles. Elles décrivent
# l'âge, le niveau d'études, la situation matrimoniale, la profession, la
# relation familiale, l'origine, le sexe, les gains et pertes en capital, le
# temps de travail hebdomadaire et le pays d'origine.

# Des valeurs manquantes sont présentes sous forme de `NaN` sur `workclass`
# (2799), `occupation` (2809) et `native-country` (857). Elles seront traitées
# dans l'étape d'audit.

## Audit & protocole — 15 pts

Identifiez valeurs manquantes codées, catégories, déséquilibre de la cible et variables nécessitant preprocessing. Construisez TRAIN/VALIDATION/TEST sans fuite.

Justifiez ce qui doit être appris sur TRAIN. Donnez une preuve numérique issue de votre pipeline.

In [3]:
# Votre travail expérimental ici
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

colonnes_numeriques = X_full.select_dtypes(include="int64").columns.tolist()
colonnes_categorielles = X_full.select_dtypes(include="category").columns.tolist()
print(colonnes_numeriques)
print(colonnes_categorielles)

na_par_colonne = X_full.isna().sum()
na_par_colonne = na_par_colonne[na_par_colonne > 0]
print(na_par_colonne)
print((na_par_colonne / len(X_full) * 100).round(2))

co_occurrence_na = X_full["workclass"].isna() & X_full["occupation"].isna()
print(co_occurrence_na.sum())
print(co_occurrence_na.sum() / X_full["workclass"].isna().sum() * 100)

cardinalites = X_full[colonnes_categorielles].nunique().sort_values(ascending=False)
print(cardinalites)

X_temp, X_test, y_temp, y_test = train_test_split(
    X_full, y_full, test_size=7326, random_state=42, stratify=y_full
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=7326, random_state=42, stratify=y_temp
)

print(X_train.shape, X_val.shape, X_test.shape)
print(y_train.value_counts(normalize=True).round(4))
print(y_val.value_counts(normalize=True).round(4))
print(y_test.value_counts(normalize=True).round(4))

pipeline_numerique = Pipeline(steps=[
    ("imputation", SimpleImputer(strategy="median")),
    ("normalisation", StandardScaler())
])

pipeline_categoriel = Pipeline(steps=[
    ("imputation", SimpleImputer(strategy="constant", fill_value="Manquant")),
    ("encodage", OneHotEncoder(handle_unknown="ignore"))
])

preprocesseur = ColumnTransformer(transformers=[
    ("numerique", pipeline_numerique, colonnes_numeriques),
    ("categoriel", pipeline_categoriel, colonnes_categorielles)
])

preprocesseur.fit(X_train)

moyenne_apprise = preprocesseur.named_transformers_["numerique"].named_steps["normalisation"].mean_
moyenne_manuelle_train = X_train[colonnes_numeriques].median().values if False else X_train[colonnes_numeriques].mean().values
moyenne_val = X_val[colonnes_numeriques].mean().values
moyenne_test = X_test[colonnes_numeriques].mean().values
moyenne_full = X_full[colonnes_numeriques].mean().values

print(np.allclose(moyenne_apprise, moyenne_manuelle_train))
print(moyenne_apprise)
print(moyenne_val)
print(moyenne_test)
print(moyenne_full)

['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']
workclass         2799
occupation        2809
native-country     857
dtype: int64
workclass         5.73
occupation        5.75
native-country    1.75
dtype: float64
2799
100.0
native-country    41
education         16
occupation        14
workclass          8
marital-status     7
relationship       6
race               5
sex                2
dtype: int64
(34190, 14) (7326, 14) (7326, 14)
class
<=50K    0.7607
>50K     0.2393
Name: proportion, dtype: float64
class
<=50K    0.7607
>50K     0.2393
Name: proportion, dtype: float64
class
<=50K    0.7607
>50K     0.2393
Name: proportion, dtype: float64
True
[3.86791167e+01 1.89609489e+05 1.00664814e+01 1.03415586e+03
 8.92556595e+01 4.04636443e+01]
[3.88080808e+01 1.90019812e+05 1.01179361e+01 1.14168769e+03
 9.12143052e+01 4.04370734e+01]
[3.83132678

In [4]:
## Audit & protocole

# La base contient 14 variables explicatives : 6 numériques (`age`, `fnlwgt`,
# `education-num`, `capital-gain`, `capital-loss`, `hours-per-week`) et 8
# catégorielles (`workclass`, `education`, `marital-status`, `occupation`,
# `relationship`, `race`, `sex`, `native-country`).

# Des valeurs manquantes codées en `NaN` sont présentes sur `workclass` (5,73%),
# `occupation` (5,75%) et `native-country` (1,75%).

# Les 2799 valeurs manquantes de `workclass` coïncident à 100% avec celles de
# `occupation`. Ce manque conjoint suggère une catégorie de non-réponse commune
# plutôt qu'un hasard, potentiellement liée à l'absence d'emploi déclaré.

# La cardinalité des variables catégorielles va de 2 (`sex`) à 41 (`native-country`),
# ce qui implique un encodage one-hot conséquent, notamment pour `native-country`.

# La cible est déséquilibrée (76,07% `<=50K`, 23,93% `>50K`). Le split
# TRAIN/VALIDATION/TEST (34190/7326/7326, soit 70/15/15) a été stratifié sur la
# cible afin de conserver ce ratio identique dans les trois jeux.

# Le preprocessing distingue un bloc numérique (imputation par la médiane puis
# normalisation) et un bloc catégoriel (imputation par la catégorie `Manquant`
# puis encodage one-hot). Ce choix d'imputation préserve le signal potentiel
# porté par la non-réponse plutôt que de l'effacer par le mode.

# La preuve de non-fuite est apportée par le `StandardScaler` : la moyenne
# apprise sur TRAIN est strictement égale à la moyenne calculée manuellement sur
# `X_train`, et diffère de VALIDATION, TEST et de la base complète (ex.
# `capital-gain` : 1034,2 sur TRAIN contre 1226,0 sur TEST).

## Quatre familles obligatoires — 25 pts

Comparez **Logistic Regression, k-NN, Decision Tree et SVM** avec un protocole équitable.

Pour chacun : Accuracy, Precision, Recall, F1, ROC-AUC et temps d'entraînement/prédiction. Ne choisissez pas le modèle final à ce stade.

In [5]:
import time
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

y_train_bin = (y_train == ">50K").astype(int)
y_val_bin = (y_val == ">50K").astype(int)
y_test_bin = (y_test == ">50K").astype(int)

modeles = {
    "logistic_regression": LogisticRegression(max_iter=1000, random_state=42),
    "knn": KNeighborsClassifier(n_neighbors=5),
    "decision_tree": DecisionTreeClassifier(random_state=42),
    "svm": CalibratedClassifierCV(
        LinearSVC(random_state=42, dual=False, max_iter=5000),
        ensemble=False
    )
}

resultats = []
pipelines_entraines = {}

for nom, modele in modeles.items():
    pipeline = Pipeline(steps=[
        ("preprocesseur", preprocesseur),
        ("modele", modele)
    ])

    debut_entrainement = time.time()
    pipeline.fit(X_train, y_train_bin)
    temps_entrainement = time.time() - debut_entrainement

    debut_prediction = time.time()
    y_pred = pipeline.predict(X_val)
    y_proba = pipeline.predict_proba(X_val)[:, 1]
    temps_prediction = time.time() - debut_prediction

    resultats.append({
        "modele": nom,
        "accuracy": accuracy_score(y_val_bin, y_pred),
        "precision": precision_score(y_val_bin, y_pred),
        "recall": recall_score(y_val_bin, y_pred),
        "f1": f1_score(y_val_bin, y_pred),
        "roc_auc": roc_auc_score(y_val_bin, y_proba),
        "temps_entrainement": temps_entrainement,
        "temps_prediction": temps_prediction
    })
    pipelines_entraines[nom] = pipeline

resultats_df = pd.DataFrame(resultats)
print(resultats_df)

                modele  accuracy  precision    recall        f1   roc_auc  \
0  logistic_regression  0.856675   0.739605  0.618939  0.673913  0.905104   
1                  knn  0.834152   0.670253  0.604107  0.635464  0.857029   
2        decision_tree  0.809992   0.598365  0.626355  0.612040  0.747055   
3                  svm  0.855856   0.741176  0.610953  0.669794  0.905861   

   temps_entrainement  temps_prediction  
0            0.426346          0.040826  
1            0.137625         12.090181  
2            2.562462          0.068075  
3            1.684354          0.062177  


In [6]:
## Quatre familles obligatoires

# Logistic Regression et SVM (noyau linéaire) obtiennent des performances quasi
# identiques (ROC-AUC 0,905 et 0,906, F1 0,674 et 0,670), suggérant que les
# classes sont proches d'être linéairement séparables dans l'espace transformé.

# k-NN a un entraînement quasi instantané (0,14s) mais une prédiction très
# lente (12,09s), car il stocke les données à l'entraînement et calcule la
# distance euclidienne entre chaque observation de VALIDATION et les 34190 de
# TRAIN uniquement au moment de la prédiction.

# Decision Tree est le modèle le plus faible, notamment en ROC-AUC (0,747),
# nettement inférieur aux trois autres. Sans limite de profondeur, l'arbre
# surapprend probablement les données de TRAIN, ce qui dégrade la qualité de
# ses probabilités.

# Le recall sur la classe minoritaire `>50K` reste modeste (0,60 à 0,63) pour
# les quatre modèles au seuil par défaut de 0,5. Comme manquer un individu
# `>50K` sera pénalisé 5 fois plus qu'un faux positif dans la section suivante,
# ce seuil ne sera probablement pas optimal pour la décision finale.

# Aucun modèle n'est choisi à ce stade : la comparaison ci-dessus sert
# uniquement à caractériser les quatre familles avant d'introduire le coût
# métier.

## Cost-sensitive decision — 20 pts

Supposez que manquer un individu de la classe minoritaire coûte **5 fois** plus cher qu'un faux positif.

Construisez votre propre coût à partir de la matrice de confusion. Comparez les quatre modèles selon ce coût.

Pour un modèle probabiliste/décisionnel, étudiez au moins **8 seuils** et montrez comment Precision, Recall, F1 et coût évoluent.

Le seuil 0,5 n'est donc pas automatiquement optimal.

In [7]:
# Votre travail expérimental ici
from sklearn.metrics import confusion_matrix

def cout_metier(y_true, y_pred, cout_fn=5, cout_fp=1):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return fp * cout_fp + fn * cout_fn

resultats_cout = []
for nom, pipeline in pipelines_entraines.items():
    y_pred = pipeline.predict(X_val)
    cout = cout_metier(y_val_bin, y_pred)
    resultats_cout.append({"modele": nom, "cout_seuil_05": cout})

resultats_cout_df = pd.DataFrame(resultats_cout)
print(resultats_cout_df)

seuils = np.linspace(0.1, 0.9, 9)
pipeline_reference = pipelines_entraines["logistic_regression"]
y_proba_reference = pipeline_reference.predict_proba(X_val)[:, 1]

resultats_seuils = []
for seuil in seuils:
    y_pred_seuil = (y_proba_reference >= seuil).astype(int)
    resultats_seuils.append({
        "seuil": seuil,
        "precision": precision_score(y_val_bin, y_pred_seuil, zero_division=0),
        "recall": recall_score(y_val_bin, y_pred_seuil, zero_division=0),
        "f1": f1_score(y_val_bin, y_pred_seuil, zero_division=0),
        "cout": cout_metier(y_val_bin, y_pred_seuil)
    })

resultats_seuils_df = pd.DataFrame(resultats_seuils)
print(resultats_seuils_df)

seuil_optimal = resultats_seuils_df.loc[resultats_seuils_df["cout"].idxmin(), "seuil"]
print(seuil_optimal)

                modele  cout_seuil_05
0  logistic_regression           3722
1                  knn           3991
2        decision_tree           4012
3                  svm           3784
   seuil  precision    recall        f1  cout
0    0.1   0.450013  0.952653  0.611274  2456
1    0.2   0.532832  0.888762  0.666239  2341
2    0.3   0.598350  0.786081  0.679487  2800
3    0.4   0.675543  0.709070  0.691901  3147
4    0.5   0.739605  0.618939  0.673913  3722
5    0.6   0.787879  0.504278  0.614957  4583
6    0.7   0.848259  0.389047  0.533438  5477
7    0.8   0.894340  0.270394  0.415243  6451
8    0.9   0.934483  0.154592  0.265296  7429
0.2


## Error audit — 15 pts

Analysez au moins **20 erreurs individuelles** de VALIDATION. Comparez FP et FN. Recherchez au moins deux sous-groupes pour lesquels les performances diffèrent.

Vous devez distinguer constat statistique, hypothèse et conclusion. N'inférez pas de causalité.

In [ ]:
# Votre travail expérimental ici


## Choix final & TEST — 15 pts

Figez modèle + preprocessing + seuil. Ouvrez TEST une seule fois. Donnez matrice de confusion, Precision, Recall, F1, ROC-AUC et coût métier.

Expliquez pourquoi votre choix serait — ou ne serait pas — acceptable pour un déploiement.

In [ ]:
# Votre travail expérimental ici


## Reproductibilité — 5 pts

### Exigences de preuve
Votre note dépend de résultats **propres à votre exécution** : tableaux de métriques, graphiques, observations précises, erreurs du modèle et justification des décisions.  
Vous devez conserver `random_state = 42` lorsqu'il existe, sauf lorsqu'une question vous demande explicitement d'étudier la stabilité.

### Ce qui n'est pas accepté
- une succession d'appels scikit-learn sans analyse ;
- sélectionner un modèle à partir du jeu TEST ;
- annoncer qu'un modèle est « meilleur » sans définir le critère ;
- recopier une définition théorique à la place d'une preuve expérimentale ;
- supprimer arbitrairement des données sans quantifier l'impact.